# SciencePlots: Publication-Ready Figures (and Maps) with Matplotlib

![SciencePlots world map](thumbnail.png)

> **Running on I-GUIDE?** Before running anything (including the `pip install` cell below), switch this notebook to the **`geoai`** conda environment: *Kernel → Change Kernel… →* select the `geoai` kernel.

Big thanks to the [SciencePlots](https://github.com/garrettj403/SciencePlots) maintainers! Sections 1–4 use the package's own [example script](https://github.com/garrettj403/SciencePlots/blob/master/examples/plot-examples.py), and section 5 applies the same styles to a map.

## Why SciencePlots?

Matplotlib's defaults are designed for the screen, not for papers. A default figure is 6.4 × 4.8 in, uses DejaVu Sans, and has a colorful default cycle that doesn't print well in grayscale. Every paper ends up with a block of `rcParams` tweaks copied from the last project.

**SciencePlots** packages those tweaks as **matplotlib style sheets**. One line changes the look of every figure:

```python
import scienceplots
plt.style.use(["science", "ieee"])
```

| Category | Styles | What they do |
|---|---|---|
| Base | `science` | Serif fonts, inward ticks on all four sides, minor ticks, thin lines, no legend frame, 3.5 × 2.6 in |
| Journals | `ieee`, `nature` | Column width and font sizes for the journal (IEEE also uses black & white line styles) |
| Colors | `bright`, `vibrant`, `muted`, `high-vis`, `high-contrast`, `retro`, `light`, `std-colors`, … | Color cycles, mostly colorblind-safe ([Paul Tol's palettes](https://sronpersonalpages.nl/~pault/)) |
| Other | `scatter`, `grid`, `notebook`, `no-latex`, `sans`, language fonts | Markers, gridlines, larger fonts for slides, and more |

Styles **stack**: later styles in the list override earlier ones.

**Plan (~15 min):** setup (2 min) → basics (2) → journal styles (3) → color cycles (2) → other styles (2) → maps (3) → wrap-up (1)

## Setup

Make sure you are in the **`geoai`** environment (see the note at the top), then install SciencePlots. It is a pure-Python package of style files, so it doesn't change any other packages.

In [ ]:
# Install SciencePlots (only needed once)
!pip install SciencePlots

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import FuncFormatter

import scienceplots  # registers the styles with matplotlib (required since v2.0)

# SciencePlots adds dozens of styles (23 of them are discrete-rainbow-1 ... -23)
print([s for s in sorted(plt.style.available) if not s.startswith(("seaborn", "_", "discrete-rainbow", "cjk"))])

### LaTeX or not?

The `science` style renders all text with **LaTeX** (`text.usetex: True`), which gives the nicest result but needs a working LaTeX install, plus `dvipng` to show figures in Jupyter. Many machines and hubs (including I-GUIDE) don't have that.

SciencePlots ships a `no-latex` style for this, but it also changes the font, which would undo the `nature` style's sans-serif font when stacked after it. So we use a small settings dict instead. It only turns LaTeX off and adds **STIXGeneral** (a Times-like font that ships with matplotlib) as a fallback font. We test whether LaTeX actually works and add the dict only when it doesn't.

In [ ]:
def latex_works():
    # Try to draw one LaTeX string; any failure means "no LaTeX here"
    try:
        with plt.style.context("science"):
            fig = plt.figure()
            fig.text(0.5, 0.5, r"$\alpha$")
            fig.canvas.draw()
        return True
    except Exception:
        return False
    finally:
        plt.close("all")


LATEX = latex_works()
NO_LATEX = [] if LATEX else [{
    "text.usetex": False,
    "font.serif": ["Times", "Times New Roman", "Nimbus Roman", "STIXGeneral", "DejaVu Serif"],
}]
print("LaTeX available:", LATEX)

---
## 1. The basics: `science`

This is the example from the SciencePlots repository: a family of curves (`model`) for different orders `p`.

In [ ]:
def model(x, p):
    return x ** (2 * p + 1) / (1 + x ** (2 * p))


pparam = dict(xlabel="Voltage (mV)", ylabel=r"Current ($\mu$A)")
x = np.linspace(0.75, 1.25, 201)


def plot_example(orders=(10, 15, 20, 30, 50, 100)):
    fig, ax = plt.subplots()
    for p in orders:
        ax.plot(x, model(x, p), label=p)
    ax.legend(title="Order")
    ax.autoscale(tight=True)
    ax.set(**pparam)
    return fig, ax

First, plain matplotlib defaults:

In [ ]:
fig, ax = plot_example()
plt.show()

Now the same code inside `plt.style.context(...)`. The context applies the style to the figures created inside the `with` block only. Use `plt.style.use(...)` instead to apply it to every figure in the notebook.

In [ ]:
with plt.style.context(["science", *NO_LATEX]):
    fig, ax = plot_example()
    print("figsize:", fig.get_size_inches())
    plt.show()

What changed: a 3.5 in wide figure (one paper column), serif fonts, ticks pointing inward on all four sides, minor ticks, thinner lines, no legend frame, and a tighter color cycle. Saving also uses `bbox_inches="tight"` automatically.

## 2. Journal styles: `ieee` and `nature`

Stack a journal style on top of `science`. **`ieee`** uses the IEEE column width, Times at 8 pt, and a **black & white** cycle (line styles instead of colors), because many IEEE papers are still read in print.

In [ ]:
with plt.style.context(["science", "ieee", *NO_LATEX]):
    fig, ax = plot_example(orders=(10, 20, 40, 100))
    fig.savefig("fig_ieee.pdf")
    plt.show()

**`nature`** uses sans-serif fonts at 7 pt, following Nature's figure guidelines.

In [ ]:
with plt.style.context(["science", "nature", *NO_LATEX]):
    fig, ax = plot_example()
    fig.savefig("fig_nature.pdf")
    plt.show()

> **Order matters:** `["science", "ieee", "std-colors"]` gives you the IEEE layout but with colors back, because `std-colors` comes last and overrides IEEE's black & white cycle.

In [ ]:
with plt.style.context(["science", "ieee", "std-colors", *NO_LATEX]):
    fig, ax = plot_example()
    plt.show()

## 3. Color cycles

SciencePlots includes several color cycles, mostly from Paul Tol's colorblind-safe palettes. Pick based on how the figure will be viewed:

- `bright`, `vibrant`, `muted`: general-purpose, colorblind-safe
- `high-contrast`: 3 colors that also stay distinct in **grayscale**
- `high-vis`: high-visibility colors plus different line styles
- `retro`, `light`, `std-colors`, `discrete-rainbow-N`: more options

In [ ]:
for cycle in ["bright", "high-contrast", "high-vis"]:
    with plt.style.context(["science", cycle, *NO_LATEX]):
        fig, ax = plot_example(orders=(10, 20, 50) if cycle == "high-contrast" else (10, 15, 20, 30, 50, 100))
        ax.set_title(f"'{cycle}'")
        plt.show()

## 4. Other styles: slides, dark mode, grids, scatter plots

- `notebook`: larger fonts and thicker lines for **presentations** and notebooks
- `dark_background` (built into matplotlib) + `science` + `high-vis`: for dark slides
- `grid`: dashed gridlines and a framed legend
- `scatter`: cycles through **marker shapes** as well as colors, with no connecting lines

In [ ]:
with plt.style.context(["science", "notebook", *NO_LATEX]):
    fig, ax = plot_example()
    plt.show()

with plt.style.context(["dark_background", "science", "high-vis", *NO_LATEX]):
    fig, ax = plot_example()
    plt.show()

The `scatter` example from the SciencePlots repo (with simpler labels, since the original legend needs LaTeX):

In [ ]:
rng = np.random.default_rng(0)

with plt.style.context(["science", "scatter", *NO_LATEX]):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.plot([-2, 2], [-2, 2], "k--")
    ax.fill_between([-2, 2], [-2.2, 1.8], [-1.8, 2.2], color="dodgerblue", alpha=0.2, lw=0)
    for i in range(7):
        x1 = rng.normal(0, 0.5, 10)
        y1 = x1 + rng.normal(0, 0.2, 10)
        ax.plot(x1, y1, label=f"#{i + 1}")
    ax.legend(loc=2, ncol=2)
    ax.set_xlabel(r"$\log_{10}(L_\mathrm{IR} / L_\odot)$")
    ax.set_ylabel(r"$\log_{10}(L_\star / L_\odot)$")
    ax.set_xlim([-2, 2])
    ax.set_ylim([-2, 2])
    plt.show()

---
## 5. Maps with SciencePlots

SciencePlots styles any matplotlib figure, so **GeoPandas maps** work too. The inward ticks on all four sides actually look like a classic map frame.

Data: [Natural Earth](https://www.naturalearthdata.com/) country boundaries (1:110m), which include 2019 population and GDP estimates. We compute **GDP per capita**.

In [ ]:
world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")
world = world[world["CONTINENT"] != "Antarctica"].copy()
world["gdp_per_capita"] = np.where(
    world["POP_EST"] > 0, world["GDP_MD"] * 1e6 / world["POP_EST"], np.nan
)
world[["NAME", "CONTINENT", "POP_EST", "GDP_MD", "gdp_per_capita"]].head()

A helper that draws the choropleth. A few map-specific choices:

- **Classed colors** (7 GDP-per-capita bands) instead of a continuous ramp, so readers can match a country to a legend class
- **Degree tick labels** (`60°W`, `30°N`), so the SciencePlots tick frame works as a lat/lon graticule
- Missing data in light gray

In [ ]:
BOUNDS = [0, 1000, 2500, 5000, 10000, 25000, 50000]
CMAP = plt.get_cmap("viridis", len(BOUNDS))  # one extra color for 50k+
NORM = BoundaryNorm(BOUNDS, CMAP.N, extend="max")

lon_fmt = FuncFormatter(lambda v, _: f"{abs(v):.0f}°{'W' if v < 0 else 'E' if v > 0 else ''}")
lat_fmt = FuncFormatter(lambda v, _: f"{abs(v):.0f}°{'S' if v < 0 else 'N' if v > 0 else ''}")


def gdp_map(ax):
    world.plot(
        ax=ax,
        column="gdp_per_capita",
        cmap=CMAP,
        norm=NORM,
        edgecolor="white",
        linewidth=0.2,
        missing_kwds={"color": "0.85"},
    )
    ax.set_xlim(-180, 180)
    ax.set_ylim(-60, 85)
    ax.set_xticks(range(-180, 181, 90))
    ax.set_yticks(range(-30, 91, 30))
    ax.xaxis.set_major_formatter(lon_fmt)
    ax.yaxis.set_major_formatter(lat_fmt)

    cb = ax.figure.colorbar(
        plt.cm.ScalarMappable(cmap=CMAP, norm=NORM),
        ax=ax,
        orientation="horizontal",
        ticks=BOUNDS,
        extend="max",
        pad=0.12,
        aspect=40,
        shrink=0.9,
    )
    cb.ax.set_xticklabels(["0", "1k", "2.5k", "5k", "10k", "25k", "50k"])
    cb.ax.minorticks_off()
    cb.set_label("GDP per capita (USD, 2019)")

Before (matplotlib defaults) and after (`science` + `ieee`). The map code is identical; only the style context and a map-shaped `figsize` change.

In [ ]:
fig, ax = plt.subplots()
gdp_map(ax)
plt.show()

with plt.style.context(["science", "ieee", *NO_LATEX]):
    fig, ax = plt.subplots(figsize=(3.3, 2.2))  # IEEE column width, map-shaped height
    gdp_map(ax)
    fig.savefig("fig_map_ieee.pdf")
    plt.show()

> **Tip:** journal styles fix the *width*, but a map's height should follow its shape. A world map in lon/lat is about 2:1, so we pass our own `figsize` with the column width and a shorter height. Pass `figsize` explicitly for any map.

A companion chart in the same style: GDP per capita vs. population by continent, using the `scatter` style so each continent gets its own **marker shape** as well as color. That keeps it readable in grayscale.

In [ ]:
with plt.style.context(["science", "scatter", *NO_LATEX]):
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for continent, group in world.dropna(subset=["gdp_per_capita"]).groupby("CONTINENT"):
        if continent == "Seven seas (open ocean)":
            continue
        ax.plot(group["POP_EST"], group["gdp_per_capita"], label=continent)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Population")
    ax.set_ylabel("GDP per capita (USD)")
    ax.legend(fontsize=6, loc="lower left", ncol=2)
    fig.savefig("fig_scatter.pdf")
    plt.show()

---
## Takeaways

- **One line** (`plt.style.use(["science", ...])`) makes every figure paper-ready: column width, serif fonts, inward ticks, colorblind-safe colors.
- **Stack styles:** base (`science`) + journal (`ieee`, `nature`) + colors (`bright`, `high-contrast`, …) + extras (`scatter`, `grid`, `notebook`). Later styles override earlier ones.
- **Works for maps:** GeoPandas plots pick up the styles too. Set a map-shaped `figsize` and label ticks in degrees.
- **LaTeX:** `science` uses LaTeX by default. Without it, use `no-latex` or turn off `text.usetex` as we did here.
- **Save as PDF** (`fig.savefig("fig.pdf")`) and include it in LaTeX without rescaling, so the fonts stay the intended size.

### Links
- SciencePlots: https://github.com/garrettj403/SciencePlots
- Style gallery (all styles): https://github.com/garrettj403/SciencePlots/wiki/Gallery
- Paul Tol's color schemes: https://sronpersonalpages.nl/~pault/
- Natural Earth data: https://www.naturalearthdata.com/